# Matrixize `rMATS` Data

## Purpose: 

Convert `rMATS` results files into matrices

## Packages and Options

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, os 
import numpy as np 
import sys

## Literals

In [2]:
cell_lines = ["HepG2", "K562"]
splice_types = ["A3SS", "A5SS", "SE", "MXE", "RI"]

# directory name to pull either batch-corrected or not batch-corrected files 
norm_or_not_pattern = "MATS_output"

# choose whether you want skipping and junction counts to be summated or not
get_total_counts = True 

# whether you want inclevel vs IJC/SJC counts
get_inclevel=False

# order of samples if junction and skipping counts are SUMMATED 
summated_sample_ordering = ["KO_Sample_1", "KO_Sample_2", "CTRL_Sample_1", "CTRL_Sample_2"]
# order of samples if junction and skipping counts are NOT SUMMATED
non_summated_sample_ordering = ["KO_Sample_1_IJC", "KO_Sample_2_IJC", "KO_Sample_1_SJC", "KO_Sample_2_SJC", "CTRL_Sample_1_IJC", "CTRL_Sample_2_IJC", "CTRL_Sample_1_SJC", "CTRL_Sample_2_SJC"]

# folder for where to output data 
output_data_folder = "../output/hg38/summated-counts/"

## Matrixization Algorithm

In [3]:
def matrixize_rmats_table(file = None, get_total_counts=None, inclevel=None):
    """
    Takes all rMATS files and extracts genomic features from them to create a matrix using junction and skipping counts. 
    
    Parameters:
        files (list): list of file paths where the rMATS files can be found (default = None)
        get_total_counts (bool): whether to sum junction and skipping counts or keep them separate (default = None)   
        inclevel (bool): if True, take the inclusion level and if False use the SJC/IJC counts 
    
    Returns: 
        matrix (pandas.DataFrame): a Pandas DataFrame where features are columns and rows are samples
        
    """
    
    assert file!=None and get_total_counts!=None and inclevel!=None
    
     # need to extract rbp and cell line from path 
    rbp_cell_line = None

    for path_part in file.split("/"): 
        # save the entire line that has the rbp-batch-cell_line nomenclature
        if "HepG2" in path_part or "K562" in path_part: 
            rbp_cell_line = path_part 

    assert rbp_cell_line != None, rbp_cell_line
    
    # dictionary where genomic position feature is key and value is sub-dict 
    # sub-dict has key for sample and value of that is the counts 
    # NOTE: if "get_total_counts=False", you will have double the sampls since "SJC" and "IJC" will be kept separate. 
    matrix_dict = {}

    # load as dataframe 
    tmp_df = pd.read_csv(file,sep="\t")

    # get index position of concatenation start string 
    # we are making strings from the exact chromosome, strand and genomic positions 
    # to do so, we need to concatenate all rows after "chr" column until the "ID.1" column 
    # this is a known assumption that all the chromosome, strand, and genomic position information 
    # is between the "chr" column upto and excluding "ID.1" column
    concatenation_start = tmp_df.columns.tolist().index("chr")
    concatenation_end = tmp_df.columns.tolist().index("ID.1")

    # take all columns to make feature name 
    # convert them to strings 
    # concatenate each column row-wise with "_" as delimiter
    # (e.g. chr17_-_62496792_62497000_62496792_62496891_62498127_62498187)
    tmp_df["Feature"] = tmp_df.iloc[
        :,concatenation_start:concatenation_end
    ].astype("str").apply(
        lambda x: '_'.join(x.values.tolist()), axis=1
    )
    
    # get inclevel columns
    if inclevel: 
        count_columns = ["IncLevel1", "IncLevel2"]
    # get IJC/SJC counts columns
    elif not inclevel: 
        count_columns = ["IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2"]
        
    # take all the counts that are comma separated per column 
    # and combine them into one column that is entirely comma-separated 
    # e.g. 780,750	758,759	260,253	543,571	 becomes 780,750,758,759,260,253,543,571
    tmp_df["Counts"] = tmp_df[count_columns].astype(str).apply(
        lambda x: ",".join(x.values.tolist()),axis=1
    )

    # subset to the columns involving features and counts 
    tmp_df = tmp_df[["Feature", "Counts"]]

    # for each feature we are now going to sum up the skipping and inclusion junction counts
    # there are 8 numbers corresponding to 4 samples
    # we are using the sample count summation indices dictionary that tells us which indices to pull from list
    for row in tmp_df.itertuples():
        
        feature = row[1]
        matrix_dict[feature] = {}
        
        numbers = row[2]
        numbers = numbers.split(',') 
        
        # get inclevel values 
        if inclevel: 
            for sample_name, ratio in zip(summated_sample_ordering, numbers): 
                
                sample_name = "{}-{}".format(rbp_cell_line, sample_name)
                
                # if NA then save as Numpy NaN
                if ratio =="NA": 
                    matrix_dict[feature][sample_name] = np.nan
                # just save the inclevel value
                else: 
                    matrix_dict[feature][sample_name] = ratio
                                
        # get actual counts (i.e. SJC/IJC)
        elif not inclevel: 

            # for each feature, we need to sum the skipping and inclusion counts 
            # 4 samples so 4 features
            if get_total_counts: 
                # get list of length 4  
                summations = summate_counts(numbers)

                assert len(summations)==4 and len(summated_sample_ordering)==4

                # assign each value in their respective order with correct sample naming
                for sample_name, value in zip(summated_sample_ordering, summations): 

                    # include RBP, batch, and cell line info in the sample name 
                    sample_name = "{}-{}".format(rbp_cell_line, sample_name)

                    matrix_dict[feature][sample_name] = value    


            # OR ELSE for each feature, separate skipping and inculsion counts 
            # 4 samples * (SJC/IJC) = 8 features 
            elif not get_total_counts:

                assert len(numbers)==8 and len(non_summated_sample_ordering)==8

                # assign each value in correct order 
                for sample_name, value in zip(non_summated_sample_ordering, numbers): 

                    # include RBP, batch, and cell line info in the sample name 
                    sample_name = "{}-{}".format(rbp_cell_line, sample_name)

                    matrix_dict[feature][sample_name] = value   
    
    # return dataframe where features are the index and columns are samples 
    return pd.DataFrame.from_dict(
        matrix_dict, 
        orient="index"
    )
    

def summate_counts(numbers): 
    """
    Parameters: 
        numbers (list): list of numbers to be summated in particular order
    
    Returns: 
        summation_list (list): list of numbers that have come as a result of correct summations
    
    """
        
    # list to save the summations 
    summation_list = []
    
    # which list indices should be pulled out for each sample
    # when summating and finding out counts per sample per feature 
    sample_count_summation_indices = {
        "KO_1": [0,2],
        "KO_2": [1,3],
        "CTRL_1": [4,6],
        "CTRL_2": [5,7],
    }
                
    # to get counts for each sample
    # pull the correct indices corresponding to junction and skipping counts for that sample 
    # sum that up and save those results in dictionary
    for sample in sample_count_summation_indices: 
        summation_list.append(
            sum(
                [int(numbers[index]) for index in sample_count_summation_indices[sample] ]
            )
        
        )
        
    return summation_list
        
        

## Validation of Matrix Creation Algorithm

In [4]:
random_file = '/project/PlatigLab/data/collaborators/BWH/encode_shRNA_processed_data/AATF-BGHLV14-HepG2/A3SS.MATS.JC.txt'

original_df = pd.read_csv(random_file, sep="\t")

original_df.iloc[111:114, :]

,ID,GeneID,geneSymbol,chr,strand,longExonStart_0base,longExonEnd,shortES,shortEE,flankingES,flankingEE,ID.1,IJC_SAMPLE_1,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,IncFormLen,SkipFormLen,PValue,FDR,IncLevel1,IncLevel2,IncLevelDifference
111,184,ENSG00000100239.15,PPP6R2,chr22,+,50419345,50419462,50419348,50419462,50416091,50416157,184,"0,0","5,3","0,1","4,9",102,99,1.000000,1.000000,"0.0,0.0","0.0,0.097",-0.049
112,185,ENSG00000100239.15,PPP6R2,chr22,+,50419345,50419462,50419348,50419462,50418866,50418979,185,"1,2","70,54","7,1","76,79",102,99,0.150101,0.845772,"0.014,0.035","0.082,0.012",-0.022
113,186,ENSG00000100239.15,PPP6R2,chr22,+,50439697,50439857,50439700,50439857,50438598,50438762,186,"4,3","79,41","7,8","67,102",102,99,0.266982,1.000000,"0.047,0.066","0.092,0.071",-0.025


In [5]:
matrixize_rmats_table(file = random_file, inclevel=False, get_total_counts=True).iloc[111:114,:]


,AATF-BGHLV14-HepG2-KO_Sample_1,AATF-BGHLV14-HepG2-KO_Sample_2,AATF-BGHLV14-HepG2-CTRL_Sample_1,AATF-BGHLV14-HepG2-CTRL_Sample_2
chr22_+_50419345_50419462_50419348_50419462_50416091_50416157,5,3,4,10
chr22_+_50419345_50419462_50419348_50419462_50418866_50418979,71,56,83,80
chr22_+_50439697_50439857_50439700_50439857_50438598_50438762,83,44,74,110


## Create All Matrices

In [ ]:
all_matrices = {}

# for each cell line 
for cell_line in cell_lines: 
    
    all_matrices[cell_line] = {}
    
    # for each splice type 
    for splice_type in splice_types: 
        
        # dataframe for outer joining 
        join_df = pd.DataFrame()
        
        "{} {}".format(cell_line, splice_type)
        
        # get all files matching cell line and splice type 
        matching_files = sorted(
            glob.glob(
                "/project/PlatigLab/data/collaborators/BWH/encode_shRNA_processed_data/*{}*/*{}*".format(cell_line, splice_type), 
                recursive=True
            )
        )
        
        # for each rMATS table
        for file in matching_files: 
            
            # convert rMATS table to matrix of counts 
            matrix = matrixize_rmats_table(
                file = file, 
                get_total_counts = get_total_counts, 
                inclevel=get_inclevel
            )
            
            if get_total_counts: 
                assert len(matrix.columns)==4
            elif not get_total_counts: 
                assert len(matrix.columns)==8
                
            join_df = join_df.join(matrix, how="outer")
        
        # check that number of samples is as expected
        if get_total_counts: 
            assert len(join_df.columns)==len(matching_files)*4
        elif not get_total_counts: 
            assert len(join_df.columns)==len(matching_files)*8
                    
        # save to dictionary 
        all_matrices[cell_line][splice_type] = join_df


'HepG2 A3SS'

'HepG2 A5SS'

'HepG2 SE'

## Find Duplicate Samples

#### Get preliminary dictionary with duplicated samples

In [ ]:
# duplicate samples dictionary 
duplicated_dict = {}

# iterate through splice types and cell lines 
# initialize necessary subdicts
for cell_line in cell_lines: 
    duplicated_dict[cell_line] = {}
    
    for splice_type in splice_types:     
        "{} {}".format(cell_line, splice_type)

        
        # transpose so that features are columns and samples are rows 
        tmp_df = all_matrices[cell_line][splice_type].T
        tmp_df.columns.size
        
        # get the features that have no missing values and save as list 
        columns_to_keep = (tmp_df.notna().sum() / tmp_df.index.size)==1
        columns_to_keep = columns_to_keep[columns_to_keep==True].index.to_list()
        
        # subset original data to these non-NaN features
        tmp_df = tmp_df[columns_to_keep]
        tmp_df.columns.size
        
        # keep all the rows that are duplicated
        tmp_df = tmp_df[tmp_df.duplicated(keep=False)]    
        
        # group the rows by all columns and get dict of row values to samples
        # save to duplicate dict 
        duplicated_dict[cell_line][splice_type]= tmp_df.groupby(list(tmp_df)).groups
        
                

### Validation of duplicated samples


Need to check whether each set of duplicated samples: 
* Are from the same batch
* Are the only set from that batch

In [ ]:
# for each cell line and splice type 
for cell_line in cell_lines:     
    for splice_type in splice_types: 
        
        # each batch per set of duplicates saved 
        all_batches = []
        
        # for each group of duplicated samples 
        for key in duplicated_dict[cell_line][splice_type]: 
            
            # get batch identifier for each sample in list 
            # and assert that there is no more than 1 unique entry 
            batch_id = list(
                set(
                    [ID.split("-")[1] for ID in duplicated_dict[cell_line][splice_type][key].to_list()]
                )
            )
            assert len(batch_id)==1, batch_id

            # append batch id to list of batch ids for each group of duplicates 
            all_batches.append(batch_id[0])
                    
        # this makes sure that between each set of duplicated samples, no batch IDs are used twice
        # this is since the value_counts() should yield a batch ID no more than twice (1 for each control sample)
        # if so, you can say that a set of duplicated samples represents batch "X" control sample "y"
        batches_across_duplicates = pd.Series(all_batches).value_counts()
        batches_across_duplicates[batches_across_duplicates>2]


**THIS MEANS THAT NOT EVERY BATCH USED 2 CONTROL SAMPLES AND HENCE, ARE NOT THE CONTROLS FOR THE ENTIRE BATCH**

Across all cell lines and splice types and for each duplicated sample group: 
* Is there the same number of them across all combinations? (There should be)
* Do they contain the same set of samples across all combinations?

In [ ]:
# for each cell line 
for cell_line in cell_lines:
    # take A5SS as an example to compare against all other splice types downstream 
    # pull out all duplicated sample groups 
    comparison_dict = duplicated_dict[cell_line]["A5SS"]
    control_duplicated_sample_groups = [comparison_dict[key].to_list() for key in comparison_dict]
    
    for splice_type in splice_types: 
        
        # make sure that the number of duplicate sample groups is the same
        assert len(comparison_dict.keys()) == len(duplicated_dict[cell_line][splice_type].keys())
        
        # for each group of duplicated samples in current iteration
        for key in duplicated_dict[cell_line][splice_type]: 
            # get list of duplicate sample group
            sample_group= duplicated_dict[cell_line][splice_type][key].to_list()
            
            # this makes sure that the exact list of duplicated sample groups is found 
            # in our "control" set of duplicated sample groups which ultimately ensures 
            # that the same samples are marked as duplicates across splice types 
            assert sample_group in control_duplicated_sample_groups 

### Visual Validation that Duplicates are Actually Duplicates

Get some samples to visually look at

In [ ]:
# transpose so that features are columns and samples are rows 
tmp_df = all_matrices["HepG2"]["A3SS"].T
tmp_df.columns.size

# get the features that have no missing values and save as list 
columns_to_keep = (tmp_df.notna().sum() / tmp_df.index.size)==1
columns_to_keep = columns_to_keep[columns_to_keep==True].index.to_list()

# subset original data to these non-NaN features
tmp_df = tmp_df[columns_to_keep]
tmp_df.columns.size

# keep all the rows that are duplicated
tmp_df = tmp_df[tmp_df.duplicated(keep=False)] 

# select 3 samples to compare 
tmp_df[
    tmp_df.index.isin(
        ['DDX52-BGHLV16-HepG2-CTRL_Sample_2','EEF2-BGHLV16-HepG2-CTRL_Sample_2','EFTUD2-BGHLV16-HepG2-CTRL_Sample_2']
    )
].iloc[:, 1:100]

## Create New Names to Label Control Samples

In [ ]:
# for each cell line, store new names for duplicate samples
new_sample_names = {}

# for each cell line
for cell_line in cell_lines:
    new_sample_names[cell_line] = {}
    
    # since we showcased that the duplicate sample groups are 
    # the same across splice types, we can just use 1 of the splice types
    # iterating through all duplicate sample groups for A3SS splice type 
    for key in duplicated_dict[cell_line]["A3SS"]: 
        
        # get RBPs involved in each group of duplicate samples
        rbps_involved = [sample.split("-")[0] for sample in duplicated_dict[cell_line]["A3SS"][key].to_list()]
        rbps_involved = "_".join(rbps_involved)

        # for each sample name
        for sample in duplicated_dict[cell_line]["A3SS"][key]: 
            
            # save new sample name as combination of RBPs involved and the rest of the suffix 
            # e.g. DDX52-BGHLV16-HepG2-CTRL_Sample_2 ---> DDX52_EEF2_EFTUD2_EIF3D_FAM120A_GEMIN5_METAP2_PA2G4_PKM2_RPS19_SERBP1_TROVE2-BGHLV16-HepG2-CTRL_Sample_2
            new_sample_names[cell_line][sample] = "{}-{}".format(
                rbps_involved, 
                "-".join(
                    sample.split("-")[1:]
                )
            )


## Update Original Matrices w/ New Sample Names

In [ ]:

# for each cell line and splice type 
for cell_line in cell_lines: 
    for splice_type in splice_types: 

        tmp_df = all_matrices[cell_line][splice_type].T
        
        tmp_df = tmp_df.rename(
            index= new_sample_names[cell_line]
        ) 

        tmp_df.index.size
        
        tmp_df = tmp_df[~tmp_df.index.duplicated(keep="first")]
        
        tmp_df.index.size
        
        tmp_df.T.to_csv(
            "{}/{}_{}.tsv.gz".format(
                output_data_folder, 
                cell_line, 
                splice_type
            ), 
            compression="gzip", 
            sep="\t"
        )
        
